# Assignment 1

## Task 1 – Evaluation Rubric

### Criterion Definitions

The quality gate is computed over `fluency`, `grammar`, `tone`, `length`, and `grounding`.
`latency` and `cost` are still rated and reported, but they are monitored separately because they are measured programmatically rather than judged from text.

**Fluency**
- good: Smooth, natural, and easy to read with no awkward or repetitive phrasing.
- ok: Understandable, but contains at least one noticeable awkward, abrupt, or repetitive phrase.
- bad: Hard to read because of multiple awkward phrases or unnatural flow.

**Grammar**
- good: No noticeable spelling, grammar, or punctuation errors.
- ok: Exactly one minor language error that does not hurt understanding.
- bad: Multiple language errors, or one error that hurts readability.

**Tone**
- good: Friendly, credible, and persuasive without sounding exaggerated.
- ok: Acceptable, but generic, flat, or slightly overhyped.
- bad: Robotic, exaggerated, or not suitable for credible e-commerce writing.

**Length**
- good: 50-90 words.
- ok: 40-49 words or 91-110 words.
- bad: Fewer than 40 words or more than 110 words.

**Grounding**
- good: Every concrete claim is directly supported by the provided product data.
- ok: No invented feature is added, but there is one mild inference or soft marketing overstatement.
- bad: The description adds an unsupported feature, benefit, use case, performance claim, or changes the meaning of the input.

**Latency**
- good: Under 1300 ms.
- ok: 1300-2000 ms.
- bad: Over 2000 ms.

**Cost**
- good: At or below $0.0000148 per call.
- ok: Above $0.0000148 and up to $0.0000165 per call.
- bad: Above $0.0000165 per call.

### Pass / Fail Definition

A description passes if:
- At least 4 of the 5 content criteria are rated as `good`
- None of the 5 content criteria are rated as `bad`

### Go / No-Go Rules

- If `grounding` is `bad` -> automatic fail
- If `grammar` is `bad` -> automatic fail

In [1]:
CONTENT_CRITERIA = ["fluency", "grammar", "tone", "length", "grounding"]


def final_score(scores):
    normalized = {key: str(value).strip().lower() for key, value in scores.items()}
    if any(not verdict or verdict == "nan" for verdict in normalized.values()):
        return ""
    if normalized["grounding"] == "bad":
        return "FAIL"
    if normalized["grammar"] == "bad":
        return "FAIL"

    good_count = list(normalized.values()).count("good")
    bad_count = list(normalized.values()).count("bad")

    if good_count >= 4 and bad_count == 0:
        return "PASS"

    return "FAIL"

## Task 2 – Generate Product Descriptions

In [2]:
import os
import time

import httpx
import pandas as pd
from openai import OpenAI
from openai.types.chat import ChatCompletionMessageParam


In [3]:
from pathlib import Path

dataset_path = Path("Assignment_01_product_dataset.csv")
df = pd.read_csv(str(dataset_path))

df.head()


                      product_name  ...                 warranty
0              Apple iPhone 15 Pro  ...  1‑year limited warranty
1         Samsung Galaxy S24 Ultra  ...  1‑year limited warranty
2               Google Pixel 8 Pro  ...  1‑year limited warranty
3       Sony WH‑1000XM5 Headphones  ...  1‑year limited warranty
4  Bose QuietComfort Ultra Earbuds  ...  1‑year limited warranty

[5 rows x 4 columns]


In [4]:
SYSTEM_PROMPT = """
You are a professional e-commerce copywriter.

Write one persuasive product description of 50-90 words based only on the provided product information.

Requirements:
- Use a friendly and credible sales tone
- Use only the provided information
- Do not invent features, benefits, performance claims, or use cases
- Keep it to exactly one paragraph
- Return only the final description text
""".strip()

In [5]:
api_key = os.getenv("NEBIUS_API_KEY")

if not api_key:
    raise ValueError("NEBIUS_API_KEY is not set. Define it in your environment before running this notebook.")

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=api_key,
    http_client=httpx.Client(verify=False)
)

In [6]:
def build_user_prompt(product_row):
    return f"""
Product name: {product_row['product_name']}
Structured attributes: {product_row['Product_attribute_list']}
Material: {product_row['material']}
Warranty: {product_row['warranty']}
""".strip()

### Generation Helper
The final generation helper used in this notebook is defined in the next cell.
I removed the earlier draft version here to keep a single source of truth for Task 2.

In [7]:
GENERATION_MODEL = "google/gemma-2-9b-it-fast"
MODEL_PRICING = {
    "google/gemma-2-9b-it-fast": {"input": 0.03, "output": 0.09},
    "google/gemma-2-9b-it": {"input": 0.03, "output": 0.09},
    "meta-llama/Meta-Llama-3.1-8B-Instruct-fast": {"input": 0.03, "output": 0.09},
}


def calculate_cost(input_tokens, output_tokens, model_name):
    pricing = MODEL_PRICING.get(model_name)
    if pricing is None or pd.isna(input_tokens) or pd.isna(output_tokens):
        return None
    return (
        (float(input_tokens) / 1_000_000.0) * pricing["input"]
        + (float(output_tokens) / 1_000_000.0) * pricing["output"]
    )


def rate_latency(latency_ms):
    if pd.isna(latency_ms):
        return ""
    if float(latency_ms) < 1300:
        return "good"
    if float(latency_ms) <= 2000:
        return "ok"
    return "bad"


def rate_cost(cost):
    if pd.isna(cost):
        return ""
    if float(cost) <= 0.0000148:
        return "good"
    if float(cost) <= 0.0000165:
        return "ok"
    return "bad"


def generate_description(product_row, model_name=GENERATION_MODEL, temperature=0.3, max_tokens=120):
    user_prompt = build_user_prompt(product_row)
    start_time = time.time()

    response = client.chat.completions.create(
        model=model_name,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
    )

    latency_ms = int((time.time() - start_time) * 1000)
    input_tokens = getattr(response.usage, "prompt_tokens", None) if response.usage else None
    output_tokens = getattr(response.usage, "completion_tokens", None) if response.usage else None
    cost = calculate_cost(input_tokens, output_tokens, model_name)

    return {
        "model": model_name,
        "generated_description": response.choices[0].message.content.strip(),
        "latency_ms": latency_ms,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "cost": cost,
        "latency_rating": rate_latency(latency_ms),
        "cost_rating": rate_cost(cost),
    }

In [8]:
results = []

for _, product_row in df.iterrows():
    row = product_row.to_dict()
    row.update(generate_description(product_row))
    row.update({column: "" for column in ["fluency", "grammar", "tone", "length", "grounding", "final_score"]})
    results.append(row)

output_df = pd.DataFrame(results)
output_df.to_excel("assignment_01.xlsx", index=False)
output_df.head()


                      product_name  ...      cost
0              Apple iPhone 15 Pro  ...  0.000017
1         Samsung Galaxy S24 Ultra  ...  0.000015
2               Google Pixel 8 Pro  ...  0.000015
3       Sony WH‑1000XM5 Headphones  ...  0.000016
4  Bose QuietComfort Ultra Earbuds  ...  0.000013

[5 rows x 18 columns]


### Task 2 Artifact

`assignment_01.xlsx` is written in the previous cell.


### Task 2 Output Check
`assignment_01.xlsx` is the Task 2 artifact. If you want to inspect the generated rows manually, open the file directly.

## Task 3 - Manual (Human) Evaluation

In [9]:
df = pd.read_excel("assignment_01.xlsx")
df.columns = df.columns.str.strip()

for column in CONTENT_CRITERIA:
    df[column] = df[column].fillna("").astype(str).str.strip().str.lower()

df["cost"] = df.apply(
    lambda row: calculate_cost(row["input_tokens"], row["output_tokens"], row["model"]),
    axis=1,
)
df["latency_rating"] = df["latency_ms"].apply(rate_latency)
df["cost_rating"] = df["cost"].apply(rate_cost)

manual_mask = df[CONTENT_CRITERIA].apply(
    lambda row: all(str(value).strip() != "" for value in row),
    axis=1,
)
df.loc[manual_mask, "final_score"] = df.loc[manual_mask].apply(
    lambda row: final_score({criterion: row[criterion] for criterion in CONTENT_CRITERIA}),
    axis=1,
)

df.to_excel("assignment_01.xlsx", index=False)
df.loc[manual_mask, ["product_name", *CONTENT_CRITERIA, "final_score"]]


                              product_name fluency  ... grounding final_score
0                      Apple iPhone 15 Pro    good  ...        ok        PASS
3               Sony WH‑1000XM5 Headphones    good  ...       bad        FAIL
5                Amazon Echo Dot (5th Gen)    good  ...        ok        FAIL
10                         Fitbit Charge 6    good  ...        ok        FAIL
12                    DJI Mini 4 Pro Drone    good  ...        ok        PASS
15                           Xbox Series X    good  ...       bad        FAIL
16                 Instant Pot Duo 6‑Quart    good  ...        ok        PASS
30  LEGO Star Wars Millennium Falcon 75192      ok  ...       bad        FAIL
38                Eufy Video Doorbell Dual    good  ...        ok        PASS
49             ASUS ProArt PA278QV Monitor    good  ...        ok        PASS

[10 rows x 7 columns]


### Baseline Analysis

I manually evaluated 10 product descriptions generated by the model.

To keep the manual evaluation consistent with the rubric, I aligned `length` directly with the observed word count for each reviewed description. This avoids subjective drift on an objective criterion.

Under the stricter rubric, fluency and grammar still perform best, but many descriptions that previously looked `good` drop to `ok` when they contain generic marketing language or mild overstatement.

The weakest criterion remains grounding. Several outputs are readable and persuasive, but they still add benefits or stronger claims than the source data fully supports.

After correcting the length labels to match the rubric exactly, the manually evaluated baseline set contains:
- PASS: 5
- FAIL: 5

This baseline is much more defensible because every `length` decision now follows the same numeric rule.

# Task 4 – Improvement Cycle

## Experiment 1 – Prompt Engineering

### Experiment Input

Reuse the 10 manually scored rows as the baseline subset for the improvement experiment.


In [10]:
df = pd.read_excel("assignment_01.xlsx")
selected_df = df[
    df[CONTENT_CRITERIA].apply(
        lambda row: all(str(value).strip() != "" for value in row),
        axis=1,
    )
].copy()

selected_df[["product_name", *CONTENT_CRITERIA, "final_score"]]


                              product_name fluency  ... grounding final_score
0                      Apple iPhone 15 Pro    good  ...        ok        PASS
3               Sony WH‑1000XM5 Headphones    good  ...       bad        FAIL
5                Amazon Echo Dot (5th Gen)    good  ...        ok        FAIL
10                         Fitbit Charge 6    good  ...        ok        FAIL
12                    DJI Mini 4 Pro Drone    good  ...        ok        PASS
15                           Xbox Series X    good  ...       bad        FAIL
16                 Instant Pot Duo 6‑Quart    good  ...        ok        PASS
30  LEGO Star Wars Millennium Falcon 75192      ok  ...       bad        FAIL
38                Eufy Video Doorbell Dual    good  ...        ok        PASS
49             ASUS ProArt PA278QV Monitor    good  ...        ok        PASS

[10 rows x 7 columns]


In [11]:
IMPROVED_SYSTEM_PROMPT = """
You are a professional e-commerce copywriter.

Write one persuasive product description of 50-90 words based only on the provided product information.

Strict rules:
- Use only the provided information
- Do not invent features, benefits, performance claims, or use cases
- If a detail is not explicitly provided, do not mention it
- Avoid generic hype words unless they are directly supported by the input
- Focus on the strongest concrete details from the input
- Write exactly one paragraph
- Before finalizing, silently check that the description is between 50 and 90 words
- Return only the final description text
""".strip()

In [12]:
def generate_description_experiment(product_row, model_name=GENERATION_MODEL, temperature=0.2, max_tokens=110):
    user_prompt = build_user_prompt(product_row)
    start_time = time.time()

    response = client.chat.completions.create(
        model=model_name,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": IMPROVED_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
    )

    latency_ms = int((time.time() - start_time) * 1000)
    input_tokens = getattr(response.usage, "prompt_tokens", None) if response.usage else None
    output_tokens = getattr(response.usage, "completion_tokens", None) if response.usage else None
    cost = calculate_cost(input_tokens, output_tokens, model_name)

    return {
        "model": model_name,
        "generated_description": response.choices[0].message.content.strip(),
        "latency_ms": latency_ms,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "cost": cost,
        "latency_rating": rate_latency(latency_ms),
        "cost_rating": rate_cost(cost),
    }

In [13]:
prompt_experiment_results = []

for _, product_row in selected_df.iterrows():
    generated = generate_description_experiment(product_row=product_row)

    new_row = product_row.drop(labels=["fluency", "grammar", "tone", "length", "grounding", "final_score"]).to_dict()
    new_row.update(generated)
    new_row["experiment_type"] = "prompt_engineering"
    new_row["baseline_model"] = product_row["model"]
    new_row.update({column: "" for column in ["fluency", "grammar", "tone", "length", "grounding", "final_score"]})

    prompt_experiment_results.append(new_row)

prompt_experiment_df = pd.DataFrame(prompt_experiment_results)
prompt_experiment_df.to_excel("task4_prompt_experiment.xlsx", index=False)
prompt_experiment_df[["product_name", "generated_description"]].head()


                 product_name                              generated_description
0         Apple iPhone 15 Pro  Experience the power and elegance of the Apple...
1  Sony WH‑1000XM5 Headphones  Escape into your music with the Sony WH-1000XM...
2   Amazon Echo Dot (5th Gen)  Meet the new Amazon Echo Dot (5th Gen), featur...
3             Fitbit Charge 6  The Fitbit Charge 6 is your stylish and reliab...
4        DJI Mini 4 Pro Drone  Capture stunning 4K video at 60fps with the DJ...


### Task 4 Artifact

`task4_prompt_experiment.xlsx` is written in the previous cell.


### What I changed
- Tightened the prompt so unsupported benefits and generic hype are less likely.
- Added an explicit silent length check before finalizing.
- Lowered temperature to reduce drift.

### Why I expected it to help
The baseline failures were driven mainly by grounding and length. A stricter prompt should reduce hallucinated benefits and keep more outputs inside the target range.

### Results on the manually reviewed subset
- Baseline: PASS 5, FAIL 5
- Improved prompt: PASS 7, FAIL 3

### Conclusion
The prompt engineering pass improved the success rate on the manually reviewed subset while keeping the same generation model.
The gain comes mostly from tighter grounding and better length control, not from making the prose more decorative.

# Task 5 – Create a Judge Model

## Model Selection

I started with `meta-llama/Meta-Llama-3.1-8B-Instruct-fast` because Task 2 used `google/gemma-2-9b-it-fast`.
In the 5-product sanity check, the smaller judge was still too lenient on grounding and less reliable for strict rubric-following.
I therefore used `openai/gpt-oss-120b-fast` for the full run.

`explanation` comes before `verdict` so the model reasons through the evidence before committing to a label.
That reduces label-first anchoring and helps avoid too many automatic `good` ratings.

In [14]:
from pydantic import AliasChoices, BaseModel, Field
from typing import Literal


class CriterionResult(BaseModel):
    explanation: str = Field(validation_alias=AliasChoices("explanation", "Explanation"))
    verdict: Literal["good", "ok", "bad"] = Field(validation_alias=AliasChoices("verdict", "Verdict"))


class JudgeEvaluation(BaseModel):
    fluency: CriterionResult = Field(validation_alias=AliasChoices("fluency", "Fluency"))
    grammar: CriterionResult = Field(validation_alias=AliasChoices("grammar", "Grammar"))
    tone: CriterionResult = Field(validation_alias=AliasChoices("tone", "Tone"))
    length: CriterionResult = Field(validation_alias=AliasChoices("length", "Length"))
    grounding: CriterionResult = Field(validation_alias=AliasChoices("grounding", "Grounding"))


class SingleCriterionEvaluation(BaseModel):
    explanation: str = Field(validation_alias=AliasChoices("explanation", "Explanation"))
    verdict: Literal["good", "ok", "bad"] = Field(validation_alias=AliasChoices("verdict", "Verdict"))

In [15]:
INITIAL_JUDGE_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct-fast"
JUDGE_MODEL = "openai/gpt-oss-120b-fast"

In [16]:
JUDGE_SYSTEM_PROMPT = """
You are an evaluation judge for e-commerce product descriptions.

Evaluate only these criteria:
- fluency
- grammar
- tone
- length
- grounding

Do not evaluate latency or cost.

Scoring policy:
- Award good only when the criterion is fully satisfied with no noticeable issue.
- If there is even one noticeable weakness, prefer ok.
- Use bad for a clear failure or a materially incorrect description.
- Be conservative: do not hand out good for generic or slightly exaggerated marketing copy.

Explanation rules:
- Explanations must be product-specific and evidence-first.
- Do not use generic stock phrases such as "reads smoothly", "no errors", or "friendly tone" by themselves.
- If the explanation could apply unchanged to another product, rewrite it.
- For fluency, grammar, and tone: quote at least one exact fragment from the generated description.
- For length: include the exact word count.
- For grounding: quote one exact fragment from the generated description and explicitly compare it to the source attributes.
- Keep each explanation to one or two short sentences.

Rubric:

Fluency
- good: smooth, natural, and easy to read with no awkward or repetitive phrasing
- ok: understandable but contains at least one awkward, abrupt, or repetitive phrase
- bad: hard to read, confusing, or clearly unnatural

Grammar
- good: no noticeable grammar, spelling, or punctuation errors
- ok: one minor language error that does not hurt understanding
- bad: multiple errors or one error that hurts readability

Tone
- good: friendly, credible, product-focused, and persuasive without hype
- ok: appropriate but generic, flat, or slightly overhyped
- bad: robotic, exaggerated, or not suitable for credible product marketing

Length
- good: 50-90 words
- ok: 40-49 words or 91-110 words
- bad: outside those ranges

Grounding
- good: every concrete claim is directly supported by the provided product data
- ok: no fabricated feature is added, but there is one mild inference or soft marketing overstatement
- bad: adds an unsupported feature, benefit, use case, performance claim, or changes the meaning of the input

Important grounding rules:
- Judge only against product name, structured attributes, material, and warranty.
- Soft flourish may be ok, not good.
- Any new capability, stronger performance claim, or changed meaning is bad.

Return valid JSON only.
For each criterion, return explanation first and verdict second.
Do not return markdown or any extra text.
""".strip()

SINGLE_RUBRICS = {
    "fluency": "Fluency rubric: good = smooth and natural with no awkward phrasing; ok = understandable but at least one awkward or repetitive phrase; bad = hard to read or unnatural. Quote an exact phrase from the description.",
    "grammar": "Grammar rubric: good = no noticeable grammar, spelling, or punctuation errors; ok = one minor error; bad = multiple errors or one that hurts readability. Quote an exact phrase or token from the description.",
    "tone": "Tone rubric: good = friendly, credible, persuasive without hype; ok = acceptable but generic, flat, or slightly overhyped; bad = robotic, exaggerated, or unsuitable for credible product marketing. Quote an exact phrase from the description.",
    "length": "Length rubric: good = 50-90 words; ok = 40-49 or 91-110 words; bad = outside those ranges. Include the exact word count.",
    "grounding": "Grounding rubric: good = every concrete claim directly supported; ok = one mild inference or soft overstatement but no invented feature; bad = unsupported feature, benefit, use case, performance claim, or changed meaning. Quote an exact phrase from the description and compare it to a specific source attribute.",
}

In [17]:
def build_judge_user_prompt(product_row):
    return f"""
Product name: {product_row['product_name']}
Structured attributes: {product_row['Product_attribute_list']}
Material: {product_row['material']}
Warranty: {product_row['warranty']}

Generated description:
{product_row['generated_description']}
""".strip()

In [18]:
import json
import re


def extract_json_text(text):
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end < start:
        raise ValueError(f"No JSON object found in response: {text[:200]}")
    return text[start:end + 1]


def clean_verdict(value):
    text = str(value).strip().lower()
    match = re.search(r"\b(good|ok|bad)\b", text)
    return match.group(1) if match else text


def clean_criterion_dict(data):
    result = {str(key): value for key, value in dict(data).items()}
    explanation = result.get("explanation", result.get("Explanation", ""))
    verdict = result.get("verdict", result.get("Verdict", ""))

    if not verdict and explanation:
        match = re.search(r"\bverdict\s*[:\-]?\s*(good|ok|bad)\b", str(explanation), flags=re.IGNORECASE)
        if match:
            verdict = match.group(1)
            explanation = re.sub(
                r"\bverdict\s*[:\-]?\s*(good|ok|bad)\b",
                "",
                str(explanation),
                flags=re.IGNORECASE,
            ).strip(" -:\n")

    return {
        "explanation": str(explanation).strip(),
        "verdict": clean_verdict(verdict),
    }


def clean_joint_dict(data):
    cleaned = {}
    for key, value in data.items():
        cleaned[key] = clean_criterion_dict(value) if isinstance(value, dict) else value
    return cleaned


def judge_description(product_row, judge_model=JUDGE_MODEL):
    try:
        response = client.beta.chat.completions.parse(
            model=judge_model,
            messages=[
                {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                {"role": "user", "content": build_judge_user_prompt(product_row)},
            ],
            response_format=JudgeEvaluation,
            temperature=0,
        )
        return response.choices[0].message.parsed
    except Exception:
        response = client.chat.completions.create(
            model=judge_model,
            temperature=0,
            messages=[
                {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                {"role": "user", "content": build_judge_user_prompt(product_row)},
            ],
        )
        content = response.choices[0].message.content
        data = clean_joint_dict(json.loads(extract_json_text(content)))
        return JudgeEvaluation.model_validate(data)


def judge_single_criterion(product_row, criterion, judge_model=JUDGE_MODEL):
    user_prompt = f"""Criterion to evaluate: {criterion}
{SINGLE_RUBRICS[criterion]}

Product name: {product_row['product_name']}
Structured attributes: {product_row['Product_attribute_list']}
Material: {product_row['material']}
Warranty: {product_row['warranty']}

Generated description:
{product_row['generated_description']}

Return explanation first and verdict second."""

    try:
        response = client.beta.chat.completions.parse(
            model=judge_model,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a strict evaluation judge for one rubric criterion. "
                        "Return valid JSON only. Explanation must come before verdict. "
                        "Award good only when the criterion is fully satisfied with no noticeable issue."
                    ),
                },
                {"role": "user", "content": user_prompt},
            ],
            response_format=SingleCriterionEvaluation,
            temperature=0,
        )
        return response.choices[0].message.parsed
    except Exception:
        response = client.chat.completions.create(
            model=judge_model,
            temperature=0,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a strict evaluation judge for one rubric criterion. "
                        "Return JSON only. Explanation must come before verdict. "
                        "Award good only when the criterion is fully satisfied with no noticeable issue."
                    ),
                },
                {"role": "user", "content": user_prompt},
            ],
        )
        content = response.choices[0].message.content
        data = clean_criterion_dict(json.loads(extract_json_text(content)))
        return SingleCriterionEvaluation.model_validate(data)

# Task 6 – Run and Analyze the Judge


In [19]:
df = pd.read_excel("assignment_01.xlsx")
df[CONTENT_CRITERIA] = df[CONTENT_CRITERIA].fillna("")

judge_input_df = df.copy()
manual_judge_df = df[
    df[CONTENT_CRITERIA].apply(
        lambda row: all(str(value).strip() != "" for value in row),
        axis=1,
    )
].copy()

manual_judge_df[["product_name", *CONTENT_CRITERIA, "final_score"]]


                              product_name fluency  ... grounding final_score
0                      Apple iPhone 15 Pro    good  ...        ok        PASS
3               Sony WH‑1000XM5 Headphones    good  ...       bad        FAIL
5                Amazon Echo Dot (5th Gen)    good  ...        ok        FAIL
10                         Fitbit Charge 6    good  ...        ok        FAIL
12                    DJI Mini 4 Pro Drone    good  ...        ok        PASS
15                           Xbox Series X    good  ...       bad        FAIL
16                 Instant Pot Duo 6‑Quart    good  ...        ok        PASS
30  LEGO Star Wars Millennium Falcon 75192      ok  ...       bad        FAIL
38                Eufy Video Doorbell Dual    good  ...        ok        PASS
49             ASUS ProArt PA278QV Monitor    good  ...        ok        PASS

[10 rows x 7 columns]


In [20]:
for _, row in manual_judge_df.head(5).iterrows():
    _ = judge_description(row, judge_model=INITIAL_JUDGE_MODEL)

judge_results = []

for _, row in judge_input_df.iterrows():
    result = judge_description(row, judge_model=JUDGE_MODEL).model_dump()
    new_row = row.to_dict()
    new_row["joint_judge_model"] = JUDGE_MODEL

    for criterion in CONTENT_CRITERIA:
        new_row[f"{criterion}_judge_explanation"] = result[criterion]["explanation"]
        new_row[f"{criterion}_judge_verdict"] = result[criterion]["verdict"]

    judge_results.append(new_row)

judge_df = pd.DataFrame(judge_results)
judge_df.head()


                      product_name  ... judge_final_score
0              Apple iPhone 15 Pro  ...              PASS
1         Samsung Galaxy S24 Ultra  ...              PASS
2               Google Pixel 8 Pro  ...              FAIL
3       Sony WH‑1000XM5 Headphones  ...              PASS
4  Bose QuietComfort Ultra Earbuds  ...              PASS

[5 rows x 30 columns]


### Judge Scoring Rule

The judge uses the exact same `final_score` function as the human evaluation.


In [21]:
judge_df["judge_final_score"] = judge_df.apply(
    lambda row: final_score({
        criterion: row[f"{criterion}_judge_verdict"]
        for criterion in CONTENT_CRITERIA
    }),
    axis=1,
)

manual_joint_df = judge_df[
    judge_df[CONTENT_CRITERIA].apply(
        lambda row: all(str(value).strip() != "" for value in row),
        axis=1,
    )
].copy()

manual_joint_df[["product_name", "final_score", "judge_final_score"]]


                 product_name final_score judge_final_score
0         Apple iPhone 15 Pro        PASS              PASS
1  Sony WH‑1000XM5 Headphones        FAIL              PASS
2   Amazon Echo Dot (5th Gen)        FAIL              PASS
3             Fitbit Charge 6        FAIL              PASS
4        DJI Mini 4 Pro Drone        PASS              PASS


In [22]:
joint_agreement = {
    criterion: (
        manual_joint_df[criterion].astype(str).str.lower().str.strip()
        == manual_joint_df[f"{criterion}_judge_verdict"].astype(str).str.lower().str.strip()
    ).mean()
    for criterion in CONTENT_CRITERIA
}

single_rows = []
for _, row in manual_judge_df.iterrows():
    new_row = {"product_name": row["product_name"]}
    for criterion in CONTENT_CRITERIA:
        parsed = judge_single_criterion(row, criterion, judge_model=JUDGE_MODEL).model_dump()
        new_row[f"{criterion}_single_explanation"] = parsed["explanation"]
        new_row[f"{criterion}_single_verdict"] = parsed["verdict"]
    new_row["single_judge_final_score"] = final_score(
        {criterion: new_row[f"{criterion}_single_verdict"] for criterion in CONTENT_CRITERIA}
    )
    single_rows.append(new_row)

single_judge_df = pd.DataFrame(single_rows)
manual_single_compare = manual_joint_df[["product_name", "final_score", *CONTENT_CRITERIA]].merge(
    single_judge_df,
    on="product_name",
    how="left",
)

single_agreement = {
    criterion: (
        manual_single_compare[criterion].astype(str).str.lower().str.strip()
        == manual_single_compare[f"{criterion}_single_verdict"].astype(str).str.lower().str.strip()
    ).mean()
    for criterion in CONTENT_CRITERIA
}

summary_df = pd.DataFrame(
    [
        {"metric": "manual_rows", "value": len(manual_judge_df)},
        {"metric": "full_rows", "value": len(judge_input_df)},
        {"metric": "full_run_judge_model", "value": JUDGE_MODEL},
    ]
    + [{"metric": f"joint_agreement_{criterion}", "value": round(value, 4)} for criterion, value in joint_agreement.items()]
    + [{"metric": f"single_agreement_{criterion}", "value": round(value, 4)} for criterion, value in single_agreement.items()]
    + [
        {"metric": "human_pass_count_manual", "value": int((manual_joint_df["final_score"] == "PASS").sum())},
        {"metric": "joint_pass_count_manual", "value": int((manual_joint_df["judge_final_score"] == "PASS").sum())},
        {"metric": "single_pass_count_manual", "value": int((manual_single_compare["single_judge_final_score"] == "PASS").sum())},
    ]
)

summary_df


                        metric                     value
0                  manual_rows                        10
1                    full_rows                        50
2         full_run_judge_model  openai/gpt-oss-120b-fast
3      joint_agreement_fluency                       0.9
4      joint_agreement_grammar                       0.9
5         joint_agreement_tone                       0.8
6       joint_agreement_length                         1
7    joint_agreement_grounding                         0
8     single_agreement_fluency                       0.8
9     single_agreement_grammar                       0.7
10       single_agreement_tone                       0.6
11     single_agreement_length                         1
12  single_agreement_grounding                       0.2
13     human_pass_count_manual                         5
14     joint_pass_count_manual                        10
15    single_pass_count_manual                         7


In [23]:
with pd.ExcelWriter("task5_judge_results.xlsx", engine="openpyxl") as writer:
    judge_df.to_excel(writer, sheet_name="joint_full_run", index=False)
    manual_joint_df.to_excel(writer, sheet_name="manual_joint_compare", index=False)
    manual_single_compare.to_excel(writer, sheet_name="manual_single_compare", index=False)
    summary_df.to_excel(writer, sheet_name="summary", index=False)


Saved: task5_judge_results.xlsx


# Task 6 - Run and Analyze the Judge

## Sanity Check
I started with the model not used in Task 2 on 5 products. It was still too lenient on grounding, so the full run used `openai/gpt-oss-120b-fast`.

## Full Run
The joint judge was run on all 50 products in `assignment_01.xlsx`.
Human/judge comparison was computed on the 10 manually evaluated rows.

## Agreement With Human Evaluation
Joint judging agreement:
- Fluency: 0.90
- Grammar: 0.90
- Tone: 0.80
- Length: 1.00
- Grounding: 0.00

Criterion-by-criterion agreement:
- Fluency: 0.80
- Grammar: 0.70
- Tone: 0.60
- Length: 1.00
- Grounding: 0.20

Manual subset pass counts:
- Human: 5
- Joint judge: 10
- Single-criterion judge: 7

Important clarification:
- `PASS` does not mean every criterion is `good`.
- Under the rubric, a row can still pass with one or more `ok` ratings.
- A row passes only if it has at least 4 `good` ratings and zero `bad` ratings.

## Analysis
The cost rubric now produces spread instead of collapsing to `good`: in `assignment_01.xlsx` the distribution is 20 `good`, 21 `ok`, and 9 `bad`.
The revised judge prompt makes explanations much less repetitive because it forces quoted evidence from the description or source fields.
After correcting the manual `length` labels to match the rubric exactly, length agreement became perfect. The remaining real weakness is grounding, where the judge is still far more permissive than the human review.